# APIM ❤️ Enterprise AI Gateway

## [Enterprise AI Gateway lab](enterprise-ai-gateway.ipynb)

End-to-end APIM gateway for LLM traffic control and MCP tool governance. Deploys APIM, Foundry (2 regions), App Insights, Log Analytics, and API Center with Terraform. Configures per-team token quotas, multi-region failover, chargeback dashboards, and governed MCP tools.

### Prerequisites
- [Python 3.12+](https://www.python.org/) installed
- [Terraform >= 1.5](https://www.terraform.io/) installed
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed
- Azure subscription with Contributor + User Access Administrator roles
- [VS Code](https://code.visualstudio.com/) with Jupyter extension

### 🚀 Get started
Proceed by running the cells below in order.

### 🗑️ Clean up resources
When finished, use the [clean-up-resources notebook](clean-up-resources.ipynb).

## 0️⃣ Initialize notebook variables

In [ ]:
import os, sys, json, time, requests

sys.path.insert(1, '../../shared')
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}"

# Azure regions
location_primary = "eastus2"
location_secondary = "swedencentral"
api_center_location = "eastus"

# Resource prefix
prefix = "aigw"

## 1️⃣ Verify Azure CLI authentication

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

## 2️⃣ Deploy infrastructure with Terraform (~15 min)

In [ ]:
terraform_variables = f'''subscription_id = "{subscription_id}"
resource_group_name = "{resource_group_name}"
location_primary = "{location_primary}"
location_secondary = "{location_secondary}"
prefix = "{prefix}"
api_center_location = "{api_center_location}"
'''

with open("terraform.tfvars", "w") as f:
    f.write(terraform_variables)

utils.print_ok("Generated terraform.tfvars")

os.environ['ARM_SUBSCRIPTION_ID'] = subscription_id
utils.run("terraform init", "Terraform initialized", "Terraform init failed")
utils.run("terraform apply -auto-approve", "Infrastructure deployed", "Terraform apply failed")

## 3️⃣ Retrieve deployment outputs

In [ ]:
def tf_output(name):
    result = utils.run(f"terraform output -raw {name}", print_command_to_run=False)
    return result.text.strip() if result.success else None

apim_gateway_url = tf_output("apim_gateway_url")
apim_name = tf_output("apim_name")
rg = tf_output("resource_group_name")
foundry_primary_endpoint = tf_output("foundry_primary_endpoint")
foundry_secondary_endpoint = tf_output("foundry_secondary_endpoint")
alpha_key = tf_output("team_alpha_subscription_key")
beta_key = tf_output("team_beta_subscription_key")
gamma_key = tf_output("team_gamma_subscription_key")

base_url = f"{apim_gateway_url}/openai/v1"

utils.print_ok(f"APIM gateway: {apim_gateway_url}")
utils.print_ok(f"APIM name: {apim_name}")
utils.print_ok(f"Foundry primary: {foundry_primary_endpoint}")
utils.print_ok(f"Foundry secondary: {foundry_secondary_endpoint}")

## 4️⃣ Import Foundry API in APIM (portal step)

This step enables the `llm-*` policy extensions. There is no CLI or IaC equivalent.

1. Azure portal → your APIM instance → APIs → **+ Add API**
2. Under "Create an AI API", click **Microsoft Foundry**
3. Select your primary Foundry account (e.g., `aigw-foundry-eus2-xxxx`)
4. Configure the API:
   - Display name: `Foundry AI Gateway`
   - Name: `foundry-ai-gateway`
   - Base path: `openai`
5. Under **Client compatibility**, select **Azure OpenAI**
6. Skip remaining tabs → **Review** → **Create**

⚠️ Run the next cell only after completing the portal import.

## 5️⃣ Post-deploy configuration

In [ ]:
api_version = "2024-05-01"

# Get APIM resource details
apim_resource = utils.run(f"az apim show -g {rg} -n {apim_name} -o json",
                          "Retrieved APIM resource", "Failed to get APIM resource")
apim_id = apim_resource.json_data['id']
system_principal_id = apim_resource.json_data['identity']['principalId']
utils.print_ok(f"APIM ID: {apim_id}")
utils.print_ok(f"System MI: {system_principal_id}")

# Discover the portal-imported API
apis_output = utils.run(f"az apim api list -g {rg} -n {apim_name} -o json",
                        "Listed APIs", "Failed to list APIs")
apis = [a for a in apis_output.json_data if a.get('path') and 'openai' in a['path']]
if not apis:
    utils.print_error("No API with 'openai' path found. Complete the portal import first.")
else:
    api = apis[0]
    api_id = api['name']
    utils.print_ok(f"Found API: {api['displayName']} (id: {api_id}, path: /{api['path']})")

    # Fix doubled path if needed
    if api['path'] != 'openai':
        utils.run(f"az apim api update -g {rg} -n {apim_name} --api-id {api_id} --set path=openai --only-show-errors",
                  "Fixed API path to /openai", "Failed to fix API path")

    # Add wildcard POST operation for v1 paths
    wildcard_body = json.dumps({
        "properties": {
            "displayName": "Wildcard POST",
            "method": "POST",
            "urlTemplate": "/{*path}",
            "description": "Pass-through for v1 and other POST paths",
            "templateParameters": [{"name": "path", "type": "string", "required": True}]
        }
    })
    import tempfile
    with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
        f.write(wildcard_body)
        tmp = f.name
    utils.run(f'az rest --method PUT --url "https://management.azure.com{apim_id}/apis/{api_id}/operations/wildcard-post?api-version={api_version}" --body @{tmp}',
              "Added wildcard POST operation", "Failed to add wildcard operation")
    os.unlink(tmp)

In [ ]:
# Grant APIM system identity RBAC on both Foundry accounts
foundry_accounts = utils.run(f"az cognitiveservices account list -g {rg} -o json",
                             "Listed Foundry accounts", "Failed to list accounts")
foundry_eus2 = next((a for a in foundry_accounts.json_data if 'eus2' in a['name']), None)
foundry_swc = next((a for a in foundry_accounts.json_data if 'swc' in a['name']), None)

rbac_role = "Cognitive Services OpenAI User"
for name, acct in [("EUS2", foundry_eus2), ("SWC", foundry_swc)]:
    if acct:
        utils.run(f'az role assignment create --assignee {system_principal_id} --role "{rbac_role}" --scope {acct["id"]} --only-show-errors',
                  f"{name} RBAC assignment OK", f"{name} RBAC assignment may have failed")

# Associate API with all three team products
for product_id in ["team-alpha", "team-beta", "team-gamma"]:
    utils.run(f"az apim product api add -g {rg} -n {apim_name} --product-id {product_id} --api-id {api_id} --only-show-errors",
              f"{product_id} ← {api_id}", f"Failed to add API to {product_id}")

# Configure API-level diagnostics
ai_logger_id = f"{apim_id}/loggers/app-insights-logger"

# Azure Monitor diagnostic (LLM log generation)
azmon_body = json.dumps({
    "properties": {
        "loggerId": f"{apim_id}/loggers/azuremonitor",
        "logClientIp": True,
        "sampling": {"samplingType": "fixed", "percentage": 100},
        "largeLanguageModel": {"logs": "enabled"}
    }
})
with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
    f.write(azmon_body)
    tmp = f.name
utils.run(f'az rest --method PUT --url "https://management.azure.com{apim_id}/apis/{api_id}/diagnostics/azuremonitor?api-version={api_version}" --body @{tmp}',
          "Azure Monitor diagnostic: LLM logs enabled", "Failed to configure azuremonitor diagnostic")
os.unlink(tmp)

# App Insights diagnostic (custom metrics)
ai_body = json.dumps({
    "properties": {
        "loggerId": ai_logger_id,
        "logClientIp": True,
        "sampling": {"samplingType": "fixed", "percentage": 100},
        "metrics": True
    }
})
with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
    f.write(ai_body)
    tmp = f.name
utils.run(f'az rest --method PUT --url "https://management.azure.com{apim_id}/apis/{api_id}/diagnostics/applicationinsights?api-version={api_version}" --body @{tmp}',
          "App Insights diagnostic: metrics enabled", "Failed to configure applicationinsights diagnostic")
os.unlink(tmp)

# Apply API-level policy
with open("policies/api-policy.xml", "r") as f:
    policy_xml = f.read()
utils.update_api_policy(subscription_id, rg, apim_name, api_id, policy_xml)
utils.print_ok("API policy applied: llm-emit-token-metric + backend pool + MI auth")

# Upgrade product policies to llm-token-limit
products = [
    ("team-alpha", 50000, "Alpha (50K TPM)"),
    ("team-beta", 20000, "Beta (20K TPM)"),
    ("team-gamma", 500, "Gamma (500 TPM)")
]
for prod_id, tpm, label in products:
    product_policy = f'''<policies>
    <inbound>
        <base />
        <llm-token-limit tokens-per-minute="{tpm}" counter-key="@(context.Subscription.Id)" estimate-prompt-tokens="false" />
    </inbound>
    <backend><base /></backend>
    <outbound><base /></outbound>
    <on-error><base /></on-error>
</policies>'''
    prod_policy_body = json.dumps({"properties": {"format": "xml", "value": product_policy}})
    with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
        f.write(prod_policy_body)
        tmp = f.name
    utils.run(f'az rest --method PUT --url "https://management.azure.com{apim_id}/products/{prod_id}/policies/policy?api-version={api_version}" --body @{tmp}',
              f"{label} → llm-token-limit applied", f"Failed to set {label} policy")
    os.unlink(tmp)

utils.print_ok("Post-deploy configuration complete")

## 6️⃣ Validate LLM gateway

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url=base_url,
    api_key=alpha_key,
    default_headers={"api-key": alpha_key}
)

response = client.chat.completions.create(
    model="gpt-51",
    messages=[{"role": "user", "content": "Say hello in one sentence."}],
    max_completion_tokens=50
)

utils.print_ok(f"Model: {response.model}")
utils.print_ok(f"Response: {response.choices[0].message.content}")
utils.print_ok(f"Tokens: {response.usage.total_tokens} (prompt: {response.usage.prompt_tokens}, completion: {response.usage.completion_tokens})")

In [ ]:
import concurrent.futures

gamma_client = OpenAI(
    base_url=base_url,
    api_key=gamma_key,
    default_headers={"api-key": gamma_key}
)

results = {"success": 0, "rate_limited": 0, "errors": 0}

def send_request(i):
    try:
        r = gamma_client.chat.completions.create(
            model="gpt-51",
            messages=[{"role": "user", "content": f"Count to {i}. Be verbose."}],
            max_completion_tokens=200
        )
        return "success"
    except Exception as e:
        if "429" in str(e):
            return "rate_limited"
        return "error"

with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(send_request, i) for i in range(10)]
    for f in concurrent.futures.as_completed(futures):
        results[f.result()] += 1

utils.print_ok(f"Success: {results['success']}, Rate limited (429): {results['rate_limited']}, Errors: {results['errors']}")
if results['rate_limited'] > 0:
    utils.print_ok("Gamma's 500 TPM quota enforced correctly")
else:
    utils.print_warning("No 429s seen. Gamma may not have hit quota yet. Try running again.")

## 7️⃣ Chargeback queries

Run these in the Azure portal after a few minutes of test traffic.

**Token usage by team** (App Insights > Logs):
```kusto
customMetrics
| where name == "Total Tokens"
| extend team = tostring(customDimensions["Subscription Name"])
| summarize TotalTokens = sum(value) by team
| order by TotalTokens desc
```

**LLM request logs** (Log Analytics > Logs):
```kusto
ApiManagementGatewayLlmLog
| project TimeGenerated, OperationName, BackendId,
          TotalTokens, PromptTokens, CompletionTokens,
          ModelName, Region
| order by TimeGenerated desc
```

**Gateway traffic** (Log Analytics > Logs):
```kusto
ApiManagementGatewayLogs
| where BackendId contains "foundry"
| summarize count() by BackendId, bin(TimeGenerated, 1m)
| render timechart
```

See `dashboards/queries.md` for more queries.

## 8️⃣ MCP tool governance (optional)

Register an external MCP server through APIM and apply governance policies.

### Register the MCP server

1. Azure portal → APIM → APIs → **MCP Servers** → **+ Create MCP server**
2. Select **Expose an existing MCP server**
3. Fill in:
   - MCP server base URL: `https://learn.microsoft.com/api/mcp`
   - Transport type: Streamable HTTP
   - Name: `microsoft-learn`
   - Base path: `learn`
4. Click **Create**
5. Select the MCP server → **MCP** → **Policies**
6. Paste the policy from `policies/mcp-governance.xml`

After registration, associate the MCP server API with the MCP product in APIM.

In [ ]:
mcp_gateway_url = f"{apim_gateway_url}/learn-mcp/mcp"

# Simple connectivity check via HTTP POST
mcp_init = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "initialize",
    "params": {
        "protocolVersion": "2025-03-26",
        "capabilities": {},
        "clientInfo": {"name": "enterprise-ai-gateway-lab", "version": "1.0.0"}
    }
}

try:
    # Get MCP subscription key if available
    mcp_key = tf_output("team_alpha_subscription_key")  # Reuse alpha key or use MCP product key
    headers = {"Content-Type": "application/json", "api-key": mcp_key, "Accept": "application/json, text/event-stream"}
    resp = requests.post(mcp_gateway_url, json=mcp_init, headers=headers, timeout=30)
    if resp.status_code == 200:
        utils.print_ok(f"MCP server reachable via APIM ({resp.status_code})")
    else:
        utils.print_warning(f"MCP returned {resp.status_code}. Ensure you completed the registration steps above.")
except Exception as e:
    utils.print_info(f"MCP server not configured yet: {e}")
    utils.print_info("Complete the registration steps in cell 8️⃣ to enable MCP governance.")

## 9️⃣ Run the full test suite

In [ ]:
os.environ['AIGW_GATEWAY_URL'] = base_url
os.environ['AIGW_ALPHA_KEY'] = alpha_key
os.environ['AIGW_BETA_KEY'] = beta_key
os.environ['AIGW_GAMMA_KEY'] = gamma_key
os.environ['AIGW_RESOURCE_GROUP'] = rg

# Derive Foundry account name for failover test
try:
    from urllib.parse import urlparse
    os.environ['AIGW_FOUNDRY_EUS2'] = urlparse(foundry_primary_endpoint).hostname.split('.')[0]
except:
    utils.print_warning("Could not derive Foundry account name. test5_failover may not work.")

utils.print_ok("Environment variables set for test suite")
utils.print_info("Run tests from terminal: cd scripts && .\\run_tests.ps1")
utils.print_info("Or run individual tests: python tests/test1_connectivity.py")

## 🗑️ Clean up resources

Use the [clean-up-resources notebook](clean-up-resources.ipynb) or run:

```bash
terraform destroy -auto-approve
```